In [1]:
import zipfile

zip_path = "archive (3).zip"   # your uploaded file name

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")   # extracts into 'data' folder

In [2]:
#set path
train_path = "data/Training"
test_path = "data/Testing"

In [3]:
#verify folder structure
import os

print(os.listdir(train_path))
print(os.listdir(test_path))

['glioma', 'meningioma', 'notumor', 'pituitary']
['glioma', 'meningioma', 'notumor', 'pituitary']


## Preprocessing

In [4]:
import numpy as np
from tqdm import tqdm
import cv2

IMG_SIZE = 224

classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img/255.0
    return img

## Load Training Data

In [5]:
X_train, y_train = [] , []

for label, category in enumerate(classes):
    folder_path = os.path.join(train_path, category)

    for img_name in tqdm(os.listdir(folder_path)):
        img_path = os.path.join(folder_path, img_name)

        #error handling
        try:
            img = preprocess_image(img_path)
            X_train.append(img)
            y_train.append(label)
        except Exception as e:
            print("Error:", img_path)
            print("Reason:", e)
#DL model require arrays not lists
X_train = np.array(X_train)
y_train = np.array(y_train)

100%|██████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:39<00:00, 35.09it/s]


In [6]:
print(X_train.shape)
print(y_train.shape)

(5600, 224, 224, 3)
(5600,)


In [7]:
print(train_path)
print(os.path.exists(train_path))

data/Training
True


In [8]:
img = cv2.imread(img_path)
if img is None:
    print("Image not readable:", img_path)

In [9]:
import os

print(os.listdir("data"))
print(os.listdir("data/Training"))

#check img exist
folder = "data/Training/glioma"
print(len(os.listdir(folder)))

print("_______________________________")

#check img loading
img_path = os.path.join(folder, os.listdir(folder)[0])

import cv2
img = cv2.imread(img_path)

print(img is None) #if true-> img not loading

['Testing', 'Training']
['glioma', 'meningioma', 'notumor', 'pituitary']
1400
_______________________________
False


## Load Testing Data

In [10]:
X_test, y_test = [],[]

for label, category in enumerate(classes):
    folder_path = os.path.join(test_path, category)

    for img_name in tqdm(os.listdir(folder_path)):
        img_path = os.path.join(folder_path, img_name)

        #error handling
        try:
            img = preprocess_image(img_path)
            X_test.append(img)
            y_test.append(label)
        except Exception as e:
            print("Error:", img_path)
            print("Reason:", e)
#DL model require arrays not lists
X_test = np.array(X_test)
y_test = np.array(y_test)

100%|████████████████████████████████████████████████████████████████████████████████| 400/400 [00:11<00:00, 35.08it/s]


In [11]:
print(X_test.shape)
print(y_test.shape)

(1600, 224, 224, 3)
(1600,)


## Convert labels

In [12]:
#One-Hot encoding
#Each label is converted into vector form
import torch
import torch.nn.functional as F

y_train = torch.tensor(y_train)
y_train = F.one_hot(y_train, num_classes=4)

In [13]:
print(y_train.shape)
print(y_train[:5])
print(y_train.dtype)

torch.Size([5600, 4])
tensor([[1, 0, 0, 0],
        [1, 0, 0, 0],
        [1, 0, 0, 0],
        [1, 0, 0, 0],
        [1, 0, 0, 0]])
torch.int64


## Create Validation Set

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size = 0.2, random_state = 42
)

In [15]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(4480, 224, 224, 3)
(1120, 224, 224, 3)
(1600, 224, 224, 3)


## Model Training

In [16]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

In [17]:
#Convert NumPy -> PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
X_val   = torch.tensor(X_val, dtype=torch.float32).permute(0, 3, 1, 2)
X_test  = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_val   = torch.tensor(y_val, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

C:\Users\dell\AppData\Local\Temp\ipykernel_5824\3470374627.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.float32)
C:\Users\dell\AppData\Local\Temp\ipykernel_5824\3470374627.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_val   = torch.tensor(y_val, dtype=torch.float32)


In [18]:
#Create DataLoader
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
val_dataset   = TensorDataset(X_val, y_val)
test_dataset  = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32)
test_loader  = DataLoader(test_dataset, batch_size=32)

In [19]:
#Load Pretrained Model (ResNet18)
model = models.resnet18(pretrained=True)

# Modify last layer
model.fc = nn.Linear(model.fc.in_features, 4)

C:\Users\dell\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\dell\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [20]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

## Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = torch.argmax(labels, dim=1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 37.1940
